# 02_data_preparation

This notebook loads player data from the SQLite database, performs comprehensive data cleaning and preparation, and prepares the dataset for statistical analysis. It includes data exploration, cleaning, feature engineering, and validation.

## Setup & Imports

Import required libraries and load player data from the SQLite database.

In [1]:
# Import required libraries
import pandas as pd
import sqlite3
import re
import os
import numpy as np
from datetime import datetime

# Ensure data directory exists
os.makedirs("data", exist_ok=True)

# Create database connection and load players data
db_path = "data/football.db"
connection = sqlite3.connect(db_path)

# Load data from SQLite database
print("Loading player data from SQLite database...")
df = pd.read_sql_query("SELECT * FROM players", connection)
connection.close()

print(f"✓ Successfully loaded {len(df)} records from database")
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Loading player data from SQLite database...
✓ Successfully loaded 351 records from database
DataFrame shape: (351, 10)
Columns: ['player_id', 'name', 'nationality', 'date_of_birth', 'team_id', 'team_name', 'position', 'market_value', 'market_value_numeric', 'market_value_tm']


## Data Exploration

Explore the structure and content of the dataset.

In [2]:
# Display first 10 rows of the DataFrame
print("First 10 rows of the dataset:")
display(df.head(10))

# Display data types
print("\nData Types:")
print(df.dtypes)

# Display basic statistics
print("\nBasic Statistics:")
display(df.describe())

# Count missing values per column
print("\nMissing Values per Column:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

First 10 rows of the dataset:


,player_id,name,nationality,date_of_birth,team_id,team_name,position,market_value,market_value_numeric,market_value_tm
0,1731,Gianluigi Donnarumma,Italy,1999-02-25,65,Manchester City FC,Goalkeeper,14.2,14.2,14.2
1,3953,Marcus Bettinelli,England,1992-05-24,65,Manchester City FC,Goalkeeper,6.4,6.4,6.4
2,153874,James Trafford,England,2002-10-10,65,Manchester City FC,Goalkeeper,5.8,5.8,5.8
3,270613,Kaden Braithwaite,England,2008-03-25,65,Manchester City FC,Defender,37.1,37.1,37.1
4,290970,Kian Noble,England,2007-02-26,65,Manchester City FC,Defender,36.4,36.4,36.4
5,292499,Floyd Samba,France,2009-01-15,65,Manchester City FC,Defender,33.2,33.2,33.2
6,180453,Sverre Nypan,Norway,2006-12-19,65,Manchester City FC,Midfielder,36.8,36.8,36.8
7,206743,Nico O'Reilly,England,2005-03-21,65,Manchester City FC,Midfielder,8.0,8.0,8.0
8,286950,Ryan McAidoo,England,2008-06-24,65,Manchester City FC,Midfielder,43.1,43.1,43.1
9,290022,Charlie Gray,England,2006-02-22,65,Manchester City FC,Midfielder,32.9,32.9,32.9



Data Types:
player_id                 int64
name                        str
nationality                 str
date_of_birth               str
team_id                   int64
team_name                   str
position                    str
market_value            float64
market_value_numeric    float64
market_value_tm         float64
dtype: object

Basic Statistics:


,player_id,team_id,market_value,market_value_numeric,market_value_tm
count,351.000000,351.000000,351.000000,351.000000,351.000000
mean,131700.213675,131.789174,15.731054,15.731054,14.625926
std,110156.382510,153.300195,22.859156,22.859156,20.844541
min,65.000000,57.000000,1.100000,1.100000,1.100000
25%,8145.500000,61.000000,4.600000,4.600000,4.800000
50%,133242.000000,65.000000,7.500000,7.500000,7.700000
75%,245838.500000,73.000000,11.950000,11.950000,10.650000
max,295284.000000,563.000000,118.000000,118.000000,118.000000



Missing Values per Column:
Series([], dtype: int64)

Total missing values: 0


## Data Cleaning

Clean and standardize the data for analysis.

In [3]:
# Define cleaning functions

def remove_duplicates(dataframe):
    """
    Remove duplicate rows from the dataset.
    
    Args:
        dataframe (pd.DataFrame): Input DataFrame
    
    Returns:
        pd.DataFrame: DataFrame with duplicates removed
    """
    # Store original count
    original_count = len(dataframe)
    
    # Remove duplicates based on player_id and team_id
    dataframe = dataframe.drop_duplicates(subset=["player_id", "team_id"], keep="first")
    
    removed_count = original_count - len(dataframe)
    print(f"✓ Removed {removed_count} duplicate rows")
    
    return dataframe

def clean_nationality(dataframe):
    """
    Clean nationality column by removing special characters and standardizing.
    
    Args:
        dataframe (pd.DataFrame): Input DataFrame
    
    Returns:
        pd.DataFrame: DataFrame with cleaned nationality
    """
    if "nationality" in dataframe.columns:
        # Remove special characters, keep only letters and spaces
        dataframe["nationality"] = dataframe["nationality"].apply(
            lambda x: re.sub(r"[^a-zA-Z\s]", "", str(x)).strip() if pd.notna(x) else x
        )
        
        # Fill missing values
        dataframe["nationality"] = dataframe["nationality"].fillna("Unknown")
        print("✓ Cleaned nationality column")
    
    return dataframe

def calculate_age(date_string):
    """
    Calculate age from date of birth string (YYYY-MM-DD format).
    
    Args:
        date_string (str): Date string in format YYYY-MM-DD
    
    Returns:
        int: Age in years, or None if invalid
    """
    if pd.isna(date_string) or date_string == "":
        return None
    
    try:
        # Parse the date string
        birth_date = pd.to_datetime(date_string)
        
        # Calculate age
        today = datetime.now()
        age = today.year - birth_date.year - ((today.month, today.day) < (birth_date.month, birth_date.day))
        
        return age
    except:
        return None

def standardize_positions(dataframe):
    """
    Standardize position names into 4 categories: Goalkeeper, Defender, Midfielder, Forward.
    
    Args:
        dataframe (pd.DataFrame): Input DataFrame
    
    Returns:
        pd.DataFrame: DataFrame with standardized positions
    """
    # Define position mappings
    position_mapping = {
        "Goalkeeper": ["Goalkeeper", "GK"],
        "Defender": ["Defender", "CB", "LB", "RB", "LWB", "RWB"],
        "Midfielder": ["Midfielder", "CM", "CDM", "CAM", "LM", "RM"],
        "Forward": ["Forward", "ST", "CF", "LW", "RW", "SS"]
    }
    
    # Function to map position to category
    def map_position(position):
        if pd.isna(position) or position == "" or position == "Unknown":
            return "Unknown"
        
        position = str(position).strip()
        
        # Search for matches in the mapping
        for category, variations in position_mapping.items():
            for variation in variations:
                if variation.lower() in position.lower():
                    return category
        
        return "Unknown"
    
    # Apply mapping
    dataframe["position_group"] = dataframe["position"].apply(map_position)
    print("✓ Standardized position names")
    
    return dataframe

# Apply cleaning functions
print("Starting data cleaning...\n")

# Remove duplicates
df = remove_duplicates(df)

# Clean nationality
df = clean_nationality(df)

# Standardize positions
df = standardize_positions(df)

print("\n✓ Data cleaning completed")

Starting data cleaning...

✓ Removed 0 duplicate rows
✓ Cleaned nationality column
✓ Standardized position names

✓ Data cleaning completed


## Feature Engineering

Create new features for analysis.

In [4]:
# Create new features

print("Starting feature engineering...\n")

# Create age column from date_of_birth
print("Creating age column...")
df["age"] = df["date_of_birth"].apply(calculate_age)

# Fill missing ages with median
if df["age"].isnull().sum() > 0:
    median_age = df["age"].median()
    df["age"] = df["age"].fillna(median_age)
    print(f"✓ Filled {df['age'].isnull().sum()} missing ages with median value: {median_age}")
else:
    print(f"✓ All ages calculated successfully")

# Create has_market_value boolean column
print("\nCreating has_market_value flag...")

# Check which market value columns exist
if "market_value_tm" in df.columns:
    df["has_market_value"] = df["market_value_tm"].notna()
elif "market_value_numeric" in df.columns:
    df["has_market_value"] = df["market_value_numeric"].notna()
else:
    df["has_market_value"] = False

has_market_value_count = df["has_market_value"].sum()
print(f"✓ {has_market_value_count} players have market value data ({100*has_market_value_count/len(df):.1f}%)")

print("\n✓ Feature engineering completed")
print(f"\nNew columns created:")
print(f"  - age: Player age in years")
print(f"  - position_group: Standardized position (4 categories)")
print(f"  - has_market_value: Boolean flag for market value availability")

Starting feature engineering...

Creating age column...
✓ All ages calculated successfully

Creating has_market_value flag...
✓ 351 players have market value data (100.0%)

✓ Feature engineering completed

New columns created:
  - age: Player age in years
  - position_group: Standardized position (4 categories)
  - has_market_value: Boolean flag for market value availability


## Data Validation

Validate the cleaned and prepared data.

In [5]:
# Validate the prepared data

print("Data Validation Results:")
print("=" * 50)

# Print final DataFrame shape
print(f"\nFinal DataFrame Shape: {df.shape}")
print(f"Total Players: {len(df)}")
print(f"Total Columns: {len(df.columns)}")

# Print value counts for position_group
print(f"\nPosition Group Distribution:")
if "position_group" in df.columns:
    position_counts = df["position_group"].value_counts()
    print(position_counts)
else:
    print("⚠ position_group column not found")

# Check for remaining missing values
print(f"\nRemaining Missing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✓ No missing values!")
else:
    print(missing[missing > 0])

# Display cleaned DataFrame sample
print(f"\nCleaned DataFrame (first 10 rows):")
display(df.head(10))

print(f"\n✓ Data validation completed successfully")

Data Validation Results:

Final DataFrame Shape: (351, 13)
Total Players: 351
Total Columns: 13

Position Group Distribution:
position_group
Unknown       218
Goalkeeper     47
Defender       30
Midfielder     30
Forward        26
Name: count, dtype: int64

Remaining Missing Values:
✓ No missing values!

Cleaned DataFrame (first 10 rows):


,player_id,name,nationality,date_of_birth,team_id,team_name,position,market_value,market_value_numeric,market_value_tm,position_group,age,has_market_value
0,1731,Gianluigi Donnarumma,Italy,1999-02-25,65,Manchester City FC,Goalkeeper,14.2,14.2,14.2,Goalkeeper,27,True
1,3953,Marcus Bettinelli,England,1992-05-24,65,Manchester City FC,Goalkeeper,6.4,6.4,6.4,Goalkeeper,34,True
2,153874,James Trafford,England,2002-10-10,65,Manchester City FC,Goalkeeper,5.8,5.8,5.8,Goalkeeper,23,True
3,270613,Kaden Braithwaite,England,2008-03-25,65,Manchester City FC,Defender,37.1,37.1,37.1,Defender,18,True
4,290970,Kian Noble,England,2007-02-26,65,Manchester City FC,Defender,36.4,36.4,36.4,Defender,19,True
5,292499,Floyd Samba,France,2009-01-15,65,Manchester City FC,Defender,33.2,33.2,33.2,Defender,17,True
6,180453,Sverre Nypan,Norway,2006-12-19,65,Manchester City FC,Midfielder,36.8,36.8,36.8,Midfielder,19,True
7,206743,Nico O'Reilly,England,2005-03-21,65,Manchester City FC,Midfielder,8.0,8.0,8.0,Midfielder,21,True
8,286950,Ryan McAidoo,England,2008-06-24,65,Manchester City FC,Midfielder,43.1,43.1,43.1,Midfielder,17,True
9,290022,Charlie Gray,England,2006-02-22,65,Manchester City FC,Midfielder,32.9,32.9,32.9,Midfielder,20,True



✓ Data validation completed successfully


## Save Cleaned Data

Store the prepared dataset in the SQLite database for use in analysis.

In [6]:
# Save cleaned data to SQLite database

print("Saving cleaned data to SQLite database...\n")

# Create database connection
connection = sqlite3.connect(db_path)

try:
    # Save cleaned DataFrame to new table
    df.to_sql("players_cleaned", connection, if_exists="replace", index=False)
    
    print(f"✓ Successfully saved {len(df)} records to 'players_cleaned' table")
    print(f"\nTable Information:")
    print(f"  - Total records: {len(df)}")
    print(f"  - Total columns: {len(df.columns)}")
    print(f"  - Columns: {list(df.columns)}")
    
    # Verify data was saved
    print(f"\nVerifying data in database...")
    verify_query = "SELECT COUNT(*) as record_count FROM players_cleaned"
    result = pd.read_sql_query(verify_query, connection)
    
    saved_count = result["record_count"][0]
    print(f"✓ Verified: {saved_count} records in players_cleaned table")
    
    print(f"\n✓ Cleaned data saved successfully to {db_path}")

except Exception as e:
    print(f"✗ Error saving data: {e}")

finally:
    # Close database connection
    connection.close()
    print("✓ Database connection closed")

Saving cleaned data to SQLite database...

✓ Successfully saved 351 records to 'players_cleaned' table

Table Information:
  - Total records: 351
  - Total columns: 13
  - Columns: ['player_id', 'name', 'nationality', 'date_of_birth', 'team_id', 'team_name', 'position', 'market_value', 'market_value_numeric', 'market_value_tm', 'position_group', 'age', 'has_market_value']

Verifying data in database...
✓ Verified: 351 records in players_cleaned table

✓ Cleaned data saved successfully to data/football.db
✓ Database connection closed
